# SEGMENTATION SNAKE

## Imports

In [1]:
import numpy as np
import cv2
import plotly.express as px
from scipy.linalg import circulant
import plotly.graph_objects as go

from utils import *

In [2]:
def createCircle(r, c, K):
    alphas = np.linspace(0, 2 * np.pi, K, endpoint=False)
    return [(int(r*np.cos(alpha)) + c[0], int(r*np.sin(alpha)) + c[1])  for alpha in alphas]

In [3]:
img = cv2.imread('imagesTP/im_goutte.png', 0)
H, W = img.shape
img = cv2.normalize(img, None, 0, 1.0, cv2.NORM_MINMAX, dtype=cv2.CV_32F)

fig = px.imshow(img, color_continuous_scale='gray')
fig.show()
K = 300

# c = list()
# cc = np.zeros((K, 1, 2))

# circle = np.array(createCircle(70, (W//2, H//2), K))

# cc[:,0,0] = circle[:,0]
# cc[:,0,1] = circle[:,1]
# c.append(cc.astype(int))

# Im_with_contours = cv2.drawContours(image=img.copy(), 
#                                     contours=c, contourIdx=-1, color=(0,255, 0), 
#                                     thickness=2, lineType=cv2.LINE_AA)

# fig = px.imshow(Im_with_contours)
# fig.show()


## ALGO SNAKE

In [ ]:
def createDMatrix(K, alpha, beta):
    c2 = np.zeros(K)
    c2[0] = -2; c2[1] = 1; c2[-1] = 1
    D2 = circulant(c2)


    c4 = np.zeros(K)
    c4[0] = 6; c4[1] = -4; c4[2] = 1;c4[-1] = -4; c4[-2] = 1
    D4 = circulant(c4)

    return alpha * D2 - beta * D4


def processSnake(img, alpha, beta, gamma, dt, initialSnake, Niter=0, snapShots=0, noise=False):
    img = img.copy()

    if noise: 
        #Si l'image est bruitée, mettre noise=True dans l'appel de la fonction
        #Ajuster les paramètres : alpha doit être plus petit (Energie externe diminuée) et Niter plus grand

        #Paramètre à ajuster en fonction du bruit
        GAUSS_KERNEL_SIZE = 11   # taille du noyau gaussien
        GAUSS_SIGMA       = 3.0  # écart-type du flou (Plus le bruit est important, plus cette variable est grande) 

        img = cv2.GaussianBlur(img, (GAUSS_KERNEL_SIZE, GAUSS_KERNEL_SIZE), GAUSS_SIGMA) 
    
    snake = np.asarray(initialSnake, dtype=float).copy()
    K = snake.shape[0]

    gradImgy, gradImgx = np.gradient(img)

    grad = gradImgx**2 + gradImgy**2
    gradNormy, gradNormx = np.gradient(grad)

    D = createDMatrix(K, alpha, beta)
    A = np.linalg.inv(np.eye(K) - dt * D)

    Niter = int(Niter)
    E = np.zeros((Niter, K, 2))

    snapshot_step = max(1, int(round(Niter * snapShots))) if snapShots else Niter + 1
    x_snap_shots = [snake[:, 0].copy()]
    y_snap_shots = [snake[:, 1].copy()]

    x = snake[:, 0].copy()
    y = snake[:, 1].copy()

    #tableaux des énergies
    e_int_tab = []
    e_ext_tab = []
    e_tot_tab = []

    # ── Paramètres de convergence ─────────────────────────────────────────────────
    CONV_THRESHOLD = 1e-3   # 0.1% de l'énergie max
    CONV_COUNT     = 5      # fois consécutives
    CONV_WINDOW    = 2000    # on compare e_tot[it] avec e_tot[it - CONV_WINDOW]

    stable_count = 0        # compteur de stabilité consécutive
    e_tot_prev   = None     # énergie totale de l'itération précédente
    e_tot_max = 0.0
    i = 0

    while i < Niter:
        xi = np.clip(x.astype(int), 0, gradNormx.shape[1] - 1)
        yi = np.clip(y.astype(int), 0, gradNormx.shape[0] - 1)

        force_x = gradNormx[yi, xi]
        force_y = gradNormy[yi, xi]

        E[i, :, 0] = D @ x - gamma * force_x
        E[i, :, 1] = D @ y - gamma * force_y

        #Energie externe
        e_ext = - gamma * np.sum(grad[yi, xi])

        #Energie élastique
        dx = x - np.roll(x, 1); dy = y - np.roll(y, 1)
        e_elastic = 0.5 * alpha * np.sum(dx**2 + dy**2)**2
        
        #Energie de courbure
        d2x = np.roll(x, -1) - 2*x + np.roll(x, 1)
        d2y = np.roll(y, -1) - 2*y + np.roll(y, 1)
        e_bending = 0.5 * beta * np.sum(d2x**2 + d2y**2)**2

        #Energie interne
        e_int = e_elastic + e_bending

        #Energie totale
        e_tot = e_int + e_ext

        e_int_tab.append(e_int)
        e_ext_tab.append(e_ext)
        e_tot_tab.append(e_tot)

        # ── Mise à jour du max courant ────────────────────────────────────────────
        e_tot_max = max(e_tot_max, abs(e_tot))

        # ── Critère de convergence ────────────────────────────────────────────────
        if i >= CONV_WINDOW:
            rel_variation = abs(e_tot - e_tot_tab[i - CONV_WINDOW]) / (e_tot_max + 1e-12)
            if rel_variation < CONV_THRESHOLD:
                stable_count += 1
                if stable_count >= CONV_COUNT:
                    print(f"Convergence atteinte à l'itération {i + 1}")
                    i += 1
                    break
        else:
            stable_count = 0

        x = A @ (x + dt * gamma * force_x)
        y = A @ (y + dt * gamma * force_y)

        if (i + 1) % snapshot_step == 0 or i == Niter - 1:
            x_snap_shots.append(x.copy())
            y_snap_shots.append(y.copy())

        i += 1

    return np.array(x_snap_shots), np.array(y_snap_shots), E, e_tot_tab, e_int_tab, e_ext_tab

In [5]:
H, W = img.shape
K = 300 # Nb points for the snake
circle = np.array(createCircle(70, (W//2, H//2), K)) # Snake initialisation

alpha = 2
beta = .02
gamma = 5
dt = .1
Niter = 20_000

x_snapshots, y_snapshots, E, e_tot, e_int, e_ext = processSnake(img, alpha, beta, gamma, dt, initialSnake=circle, Niter=Niter, snapShots=1/100)
showAnim(img, x_snapshots, y_snapshots, 100)
plot_energy(e_int, e_ext, e_tot)
displayGradEnergy(E)

Convergence atteinte à l'itération 7970


array([2.00276773, 1.22281975, 0.77021301, ..., 0.        , 0.        ,
       0.        ], shape=(20000,))